# Urban Heat & Cooling-Priority Mapping — Track B / SB1: Validation Sample Generation

**NUS-ISS Practice Module, Week 2.** One notebook, run top to bottom:

1. **Setup** — install deps, authenticate Earth Engine, mount Drive (all auth up front)
2. **SB1.1** — config (AOI, season window — locked to match Track A's C4 decision)
3. **SB1.2** — fetch the real Singapore boundary from data.gov.sg (URA subzones
   dissolved into one polygon) — sampling is restricted to this, not a loose
   bounding box, so points can't land in Malaysia/Indonesia territory or open sea
4. **SB1.3** — cloud masking + season-filter helpers
5. **SB1.4** — Sentinel-2 composite + valid-data mask (season-controlled, C4)
6. **SB1.5** — load WorldCover, collapse to the 4-class scheme (vegetation /
   built_up / bare / water), mask to valid-data footprint (CRS checkpoint)
7. **SB1.6** — coverage check
8. **SB1.7** — class area histogram (pulled once, counted locally)
9. **SB1.8** — sample allocation (vegetation oversample, floor, renormalize to 200)
10. **SB1.9** — stratified sampling draw + shortfall check
11. **SB1.10** — verdict (checks dict, PASS/FAIL)
12. **SB1.11** — build labeling table (point_id, lon/lat, class name, empty label columns)
13. **SB1.12** — export to Drive (GeoJSON + CSV)
14. **SB1.13 (optional)** — reload check from mounted Drive

Run cells in order. SB1.4 depends on SB1.2-SB1.3, SB1.5 depends on SB1.4, SB1.9 depends
on SB1.7-SB1.8. Output feeds the joint labeling session (Week 2, Step 2) and, once
labeled, becomes the validation set for the RF/U-Net/ensemble comparison (Step 3-4).

**Scope note:** this notebook produces the *validation* sample only. Per the locked
label protocol, ESA WorldCover here is used purely to stratify *which points get
picked* for hand-labeling — it is never the answer key. The `worldcover_class`
column carried into the output is reference context for the labeler, not a label.

**Class scheme (gate-review cutdown):** WorldCover's native classes are collapsed
to 4 buckets — **vegetation** (tree/shrub/grass/crop/mangrove/wetland/moss),
**built_up**, **bare**, **water** — matching `label_points_interactive.ipynb`'s
labeling dropdown exactly, so no separate collapse step is needed downstream.
Water is a genuine stratified class here (not excluded) since it's one of the
4 official classes; only snow/ice is masked out (not present in Singapore).
The original fine-grained WorldCover class is still carried through as
`worldcover_class_raw`/`worldcover_class_raw_name` for audit-trail context.

**Composite note:** SB1.3 can either rebuild a lightweight Sentinel-2 composite
inline (for deriving a valid-data mask only — no bands from it are used as
classifier input here) or point at your already-built, verified GEE composite
asset if you'd rather reuse its mask directly. See the `USE_EXISTING_COMPOSITE`
flag in SB1.1.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [3]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q earthengine-api geemap pandas geopandas requests


## Setup 2 — Authenticate & initialize Earth Engine

In [4]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- must match the project used in
                                      #     Week-1 gates and Track A's notebooks

if PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID is still the placeholder. Set it to your actual GCP project ID "
        "(lowercase-with-hyphens, from console.cloud.google.com), then re-run this cell."
    )

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


## Setup 3 — Mount Google Drive

In [5]:
# --- SETUP CELL 3: Mount Google Drive (needed to write/read the export) ----
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 4 — Initialize shared results tracker

In [6]:
# --- SETUP CELL 4: Initialize shared results tracker ------------------------
# SB1.9's verdict cell writes into this dict, same pattern as Week-1's
# gate_results and Track A's track_a_results, so this notebook's outcome
# can be compiled/compared the same way.
track_b_results = {}
print("track_b_results initialized")


track_b_results initialized


---
# SB1 — Stratified validation sample (200 points)

Draws a stratified random sample of points restricted to Singapore's real
land boundary (not a bounding box), stratified on WorldCover class
proportions with vegetation classes oversampled, restricted to a valid
Sentinel-2 data footprint with water excluded.


## SB1.1 — Config (AOI, season window, sampling params)

In [7]:
# --- SB1 CELL 1: Config ------------------------------------------------------
# NOTE: this assumes SETUP CELL 2 (EE auth) has already run in this session.

# Singapore bounding box — same AOI convention as Track A's gee_heat_variants.ipynb.
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

# Season-controlled, multi-year window (C4) — LOCKED, identical to
# gee_heat_variants.ipynb and adaptive_capacity_pillar.ipynb. Do not drift
# this independently: if the greenery layer and the heat layer are built
# from different date windows, any correlation between them in the final
# score is partly an artifact of season mismatch, not real signal.
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
DRY_SEASON_MONTHS = [4, 5, 10, 11]      # both inter-monsoon periods — see
                                         # gee_heat_variants.ipynb SB1.1 for the
                                         # season_window_diagnostic.ipynb justification
S2_CLOUD_PROB_MAX = 70                  # s2cloudless threshold, same as Track A

TARGET_SCALE = 10
S2_UTM_CRS = "EPSG:32648"               # UTM Zone 48N — same explicit-projection
                                         # discipline as Track A (median() composites
                                         # don't carry a real projection by default;
                                         # this is what silently broke their bicubic10
                                         # variant, so we set it explicitly here too)

# If you already have a verified Sentinel-2 composite asset from your Week-1
# build, point at it here to reuse its valid-data mask directly rather than
# rebuilding a second composite that could drift from the one your
# classifiers actually train on. Leave as None to rebuild inline (SB1.3).
USE_EXISTING_COMPOSITE = False
EXISTING_COMPOSITE_ASSET = None   # e.g. "projects/nus-iss-urban-heat-sg/assets/s2_composite_2021_2026"

# WorldCover — CONFIRM this matches whatever version/year your training
# labels use (per the locked label protocol: WorldCover for training,
# Dynamic World + hand-labels for validation). A version mismatch here
# would stratify the validation sample on different class boundaries than
# training was built on.
#
# NOTE ON DATES: 2020 (v100) and 2021 (v200) are the only WorldCover
# releases that exist -- there is no newer global release to switch to.
# This asset's role here is ONLY to set sampling proportions (which
# points get drawn into the 300-point sample) -- it is NOT the date you
# label against. When you and your teammate assign agreed_label during
# the joint session, label against CURRENT/recent basemap imagery, not
# 2021 -- the deliverable validates the classifier's present-day
# accuracy, so ground truth needs to reflect present-day land cover.
WORLDCOVER_ASSET = "ESA/WorldCover/v200/2021"

# WorldCover v200 native class codes
WC_TREE, WC_SHRUB, WC_GRASS, WC_CROP = 10, 20, 30, 40
WC_BUILTUP, WC_BARE, WC_SNOWICE, WC_WATER = 50, 60, 70, 80
WC_WETLAND, WC_MANGROVE, WC_MOSSLICHEN = 90, 95, 100

# Original fine-grained names — kept only for the audit-trail columns in
# the final output (worldcover_class_raw_name), NOT used for stratification
# anymore now that classes are collapsed to the 4-class gate-review scheme.
ORIGINAL_WC_NAMES = {
    WC_TREE: "tree_cover", WC_SHRUB: "shrubland", WC_GRASS: "grassland",
    WC_CROP: "cropland", WC_BUILTUP: "built_up", WC_BARE: "bare_sparse_veg",
    WC_SNOWICE: "snow_ice", WC_WATER: "water", WC_WETLAND: "herbaceous_wetland",
    WC_MANGROVE: "mangroves", WC_MOSSLICHEN: "moss_lichen",
}

# --- 4-class gate-review scheme (matches label_points_interactive.ipynb's
# CLASS_OPTIONS/COLLAPSE_MAP exactly — keep these in sync if either changes) ---
BUCKET_VEGETATION, BUCKET_BUILTUP, BUCKET_BARE, BUCKET_WATER = 1, 2, 3, 4
BUCKET_NAMES = {
    BUCKET_VEGETATION: "vegetation", BUCKET_BUILTUP: "built_up",
    BUCKET_BARE: "bare", BUCKET_WATER: "water",
}

# Remap table: raw WorldCover code -> bucket id. Snow/ice maps to sentinel 0
# (masked out in SB1.5 — not present in Singapore, not one of the 4 classes).
WC_TO_BUCKET_FROM = [WC_TREE, WC_SHRUB, WC_GRASS, WC_CROP, WC_BUILTUP,
                      WC_BARE, WC_SNOWICE, WC_WATER, WC_WETLAND,
                      WC_MANGROVE, WC_MOSSLICHEN]
WC_TO_BUCKET_TO = [BUCKET_VEGETATION, BUCKET_VEGETATION, BUCKET_VEGETATION,
                    BUCKET_VEGETATION, BUCKET_BUILTUP, BUCKET_BARE, 0,
                    BUCKET_WATER, BUCKET_VEGETATION, BUCKET_VEGETATION,
                    BUCKET_VEGETATION]

# Oversample the vegetation bucket relative to its raw area share — this is
# the pillar the cooling-priority score actually consumes, so worth extra
# representation even after collapsing subtypes away. If "bare" turns out
# too thin in practice (it was the smallest class in an earlier unstratified
# test run — 13/200), add BUCKET_BARE to this list too.
OVERSAMPLE_BUCKETS = [BUCKET_VEGETATION]

# Sampling params
TOTAL_POINTS = 200
VEGETATION_OVERSAMPLE_FACTOR = 2.0   # applied to OVERSAMPLE_BUCKETS before renormalizing
MIN_POINTS_PER_CLASS = 8             # floor so rare-but-present classes aren't zeroed out
RANDOM_SEED = 42                     # pinned, per eval rules

COVERAGE_THRESHOLD = 0.90

EXPORT_DESCRIPTION = "validation_sample_200"
EXPORT_FOLDER = "urban_heat_sg"
EXPORT_FILE_PREFIX = "validation_sample_200"

print(f"AOI: Singapore bbox (prefilter only — real sampling boundary built in SB1.2)")
print(f"Season window: months {DRY_SEASON_MONTHS} across years {YEARS}")
print(f"WorldCover asset: {WORLDCOVER_ASSET}")
print(f"Target: {TOTAL_POINTS} points, vegetation oversample x{VEGETATION_OVERSAMPLE_FACTOR}, "
      f"floor {MIN_POINTS_PER_CLASS}/class")


AOI: Singapore bbox (prefilter only — real sampling boundary built in SB1.2)
Season window: months [4, 5, 10, 11] across years [2021, 2022, 2023, 2024, 2025, 2026]
WorldCover asset: ESA/WorldCover/v200/2021
Target: 200 points, vegetation oversample x2.0, floor 8/class


## SB1.2 — Fetch the real Singapore boundary (not a loose bounding box)

`sg_bbox` above is only used as a cheap prefilter for `filterBounds()` on image
collections. The actual sampling region is `sg_boundary` below — Singapore's
real administrative extent, built by dissolving URA subzone polygons fetched
directly from data.gov.sg (same dataset ID and fetch pattern as Track A's
`gee_heat_variants.ipynb` S2.2, including the same dotted-property-name
sanitization that fixed their EE upload bug). This keeps sampled points off
open sea and out of neighboring-country slivers (Johor to the north, Batam/
Bintan to the south) that a rectangular bbox would otherwise include.


In [8]:
# --- SB1 CELL 2: Fetch Singapore boundary from data.gov.sg ------------------
import requests
import json as _json

SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"  # MP19 Subzone Boundary (No Sea), GEOJSON
SUBZONE_LOCAL_PATH = "/content/ura_subzones.geojson"

def fetch_datagovsg_geojson(dataset_id, out_path):
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    download_url = payload["data"]["url"]
    geojson_bytes = requests.get(download_url).content
    with open(out_path, "wb") as f:
        f.write(geojson_bytes)
    return out_path

try:
    subzone_path = fetch_datagovsg_geojson(SUBZONE_DATASET_ID, SUBZONE_LOCAL_PATH)
    print(f"Downloaded subzones GeoJSON -> {subzone_path}")
except Exception as e:
    print(f"⚠️  Auto-download failed ({e}). Manual fallback:")
    print("   1. Download GeoJSON from https://data.gov.sg/datasets/d_8594ae9ff96d0c708bc2af633048edfb/view")
    print(f"   2. Upload it to {SUBZONE_LOCAL_PATH} in the Colab file browser (left sidebar)")
    print("   3. Re-run this cell — it will find the file and skip the failed download.")
    raise

with open(subzone_path) as f:
    subzone_geojson = _json.load(f)

n_features_local = len(subzone_geojson.get("features", []))
print(f"Parsed GeoJSON locally: {n_features_local} features")
if n_features_local == 0:
    raise ValueError("Downloaded GeoJSON has zero features — check the file/download before proceeding.")

# EE rejects property names containing '.' (ArcGIS-style SHAPE.AREA / SHAPE.LEN
# fields) — sanitize locally before upload. Same fix as Track A's S2.2.
for feature in subzone_geojson.get("features", []):
    props = feature.get("properties", {})
    for old_key in list(props.keys()):
        if "." in old_key:
            props[old_key.replace(".", "_")] = props.pop(old_key)

subzones = ee.FeatureCollection(subzone_geojson)
n_subzones = subzones.size().getInfo()
print(f"Loaded {n_subzones} subzones into an ee.FeatureCollection")

# Dissolve all subzone polygons into one boundary geometry — this is the real
# sampling region, used everywhere below instead of sg_bbox.
sg_boundary = subzones.union(1).first().geometry()
boundary_area_km2 = sg_boundary.area(1).divide(1e6).getInfo()
print(f"Dissolved Singapore boundary built. Approx area: {boundary_area_km2:,.1f} km²")
print("(Sanity check: Singapore's land area is ~730-735 km² — if this is wildly")
print(" off, check the subzone fetch above before trusting sg_boundary downstream.)")


Downloaded subzones GeoJSON -> /content/ura_subzones.geojson
Parsed GeoJSON locally: 332 features
Loaded 332 subzones into an ee.FeatureCollection
Dissolved Singapore boundary built. Approx area: 788.3 km²
(Sanity check: Singapore's land area is ~730-735 km² — if this is wildly
 off, check the subzone fetch above before trusting sg_boundary downstream.)


## SB1.3 — Cloud masking + season-filter helpers

In [9]:
# --- SB1 CELL 2: Cloud masking + season-filter helpers ----------------------
def mask_s2_clouds(cloud_prob_image):
    """s2cloudless probability mask for Sentinel-2 L2A. Same helper as
    Track A's gee_heat_variants.ipynb SB1.4."""
    return cloud_prob_image.select("probability").lt(S2_CLOUD_PROB_MAX)


def date_filter_for_years_months(collection, years, months):
    """Union filter: keep images that fall in ANY (year, month) combo. Used
    to build season-controlled, multi-year composites (C4) — a plain
    filterDate(start, end) would mix wet/dry-season conditions. Identical
    logic to Track A's gee_heat_variants.ipynb."""
    filters = []
    for y in years:
        for m in months:
            start = ee.Date.fromYMD(y, m, 1)
            end = start.advance(1, "month")
            filters.append(ee.Filter.date(start, end))
    return collection.filter(ee.Filter.Or(*filters))


print("Cloud masking + season-filter helpers defined.")


Cloud masking + season-filter helpers defined.


## SB1.4 — Sentinel-2 composite + valid-data mask (season-controlled, C4)

In [10]:
# --- SB1 CELL 4: Sentinel-2 composite + valid-data mask ---------------------
if USE_EXISTING_COMPOSITE:
    if not EXISTING_COMPOSITE_ASSET:
        raise ValueError("USE_EXISTING_COMPOSITE is True but EXISTING_COMPOSITE_ASSET is not set.")
    composite = ee.Image(EXISTING_COMPOSITE_ASSET).clip(sg_boundary)
    print(f"Loaded existing composite asset: {EXISTING_COMPOSITE_ASSET}")
else:
    s2_sr = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(sg_bbox)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    )
    s2_sr = date_filter_for_years_months(s2_sr, YEARS, DRY_SEASON_MONTHS)
    print("Sentinel-2 scenes after season filter (pre-mask):", s2_sr.size().getInfo())

    s2_cloud_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(sg_bbox)
    s2_cloud_prob = date_filter_for_years_months(s2_cloud_prob, YEARS, DRY_SEASON_MONTHS)

    joined = ee.Join.saveFirst("cloud_mask").apply(
        primary=s2_sr,
        secondary=s2_cloud_prob,
        condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
    )

    def _mask(img):
        img = ee.Image(img)
        cloud_img = ee.Image(img.get("cloud_mask"))
        clear_mask = mask_s2_clouds(cloud_img)
        return img.updateMask(clear_mask)

    s2_masked = ee.ImageCollection(joined).map(_mask)
    print("Sentinel-2 usable scenes (post-mask):", s2_masked.size().getInfo())

    composite = s2_masked.select("B4").median().clip(sg_boundary)

# Explicit reprojection before anything downstream relies on this as a real
# fine-resolution grid — same discipline as Track A's lst_30m fix (a
# .median() composite's default projection is degenerate, not a usable
# 10m UTM grid, until you set one explicitly).
composite = composite.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE)
valid_mask = composite.mask()

proj_check = composite.projection().getInfo()
print(f"Composite projection: {proj_check['crs']} (expect {S2_UTM_CRS})")


Sentinel-2 scenes after season filter (pre-mask): 59
Sentinel-2 usable scenes (post-mask): 59
Composite projection: EPSG:32648 (expect EPSG:32648)


## SB1.5 — Load WorldCover, collapse to 4-class scheme, mask to valid-data footprint

CRS checkpoint, same discipline as Track A's zonal join step: explicitly
reproject WorldCover onto the same UTM grid as the composite before masking,
rather than trusting its native EPSG:4326 default to line up implicitly with
`valid_mask`. Also clipped to `sg_boundary`, not `sg_bbox` — see SB1.2.

WorldCover's native classes are collapsed to the 4-class gate-review scheme
via `remap()` — snow/ice maps to a sentinel value and gets masked out (not
present in Singapore); everything else maps to vegetation/built_up/bare/water.
The raw WorldCover code is kept as a second band (`wc_raw`) purely so the
final output can carry an audit-trail column — it plays no role in
stratification.


In [11]:
# --- SB1 CELL 5: Load WorldCover, collapse to 4 classes, CRS checkpoint -----
worldcover_raw = ee.Image(WORLDCOVER_ASSET).select("Map")

wc_native_proj = worldcover_raw.projection().getInfo()
print(f"WorldCover native projection: {wc_native_proj['crs']}")
print(f"Composite projection: {S2_UTM_CRS}")
print("(Reprojecting WorldCover onto the composite's grid explicitly below —")
print(" don't assume reduceRegion/stratifiedSample will silently reconcile these.)")

worldcover = worldcover_raw.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE).clip(sg_boundary)

# Collapse to the 4-class bucket scheme. Pixels not in WC_TO_BUCKET_FROM (or
# explicitly mapped to sentinel 0, i.e. snow/ice) get masked out below.
wc_bucket = worldcover.remap(WC_TO_BUCKET_FROM, WC_TO_BUCKET_TO, 0).rename("wc_class")
wc_bucket = wc_bucket.updateMask(wc_bucket.neq(0))

combined_mask = wc_bucket.mask().And(valid_mask)

wc_sampling_frame = (
    ee.Image.cat([wc_bucket, worldcover.rename("wc_raw")])
    .updateMask(combined_mask)
    .clip(sg_boundary)
)

print("WorldCover collapsed to 4-class scheme (vegetation/built_up/bare/water),")
print("masked to valid-data footprint, clipped to the real Singapore boundary.")
print("Note: water is now a genuine stratified class (not excluded) — only")
print("snow/ice is masked out, since it doesn't occur in Singapore.")


WorldCover native projection: EPSG:4326
Composite projection: EPSG:32648
(Reprojecting WorldCover onto the composite's grid explicitly below —
 don't assume reduceRegion/stratifiedSample will silently reconcile these.)
WorldCover collapsed to 4-class scheme (vegetation/built_up/bare/water),
masked to valid-data footprint, clipped to the real Singapore boundary.
Note: water is now a genuine stratified class (not excluded) — only
snow/ice is masked out, since it doesn't occur in Singapore.


## SB1.6 — Coverage check

Same purpose as Track A's S2.6 coverage check: confirm there's enough valid
pixel coverage in the sampling frame before spending compute on the
histogram + stratified draw downstream.


In [12]:
# --- SB1 CELL 6: Coverage check -----------------------------------------------
def coverage_fraction(image, aoi, scale):
    stats = image.mask().reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True,
    )
    return ee.Number(stats.get(list(stats.getInfo().keys())[0]))

sampling_frame_coverage = coverage_fraction(wc_sampling_frame.select("wc_class"), sg_boundary, TARGET_SCALE).getInfo()
print(f"Sampling-frame valid-pixel coverage over Singapore boundary: {sampling_frame_coverage*100:.1f}%")

coverage_ok = sampling_frame_coverage >= COVERAGE_THRESHOLD

if coverage_ok:
    print(f"\n✅ Coverage OK (>= {COVERAGE_THRESHOLD*100:.0f}%). Proceeding to histogram.")
else:
    print(f"\n⚠️  Coverage below {COVERAGE_THRESHOLD*100:.0f}% threshold:")
    print("   - Widen DRY_SEASON_MONTHS or YEARS in SB1.1")
    print("   - Raise S2_CLOUD_PROB_MAX (fewer scenes get masked out)")
    print("   - If using USE_EXISTING_COMPOSITE, confirm that asset's own coverage first")


Sampling-frame valid-pixel coverage over Singapore boundary: 99.9%

✅ Coverage OK (>= 90%). Proceeding to histogram.


## SB1.7 — Class area histogram (pulled once, counted locally)

In [13]:
# --- SB1 CELL 7: Class area histogram ----------------------------------------
# Single reduceRegion + single .getInfo() — same "pull once" discipline as
# Track A's zonal join (repeated .getInfo() calls on a lazy EE graph
# re-run the whole computation each time, not just the final aggregation).
class_hist_raw = wc_sampling_frame.select("wc_class").reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=sg_boundary,
    scale=TARGET_SCALE,
    maxPixels=1e13,
    bestEffort=True,
).get("wc_class").getInfo()

class_hist = {int(k): v for k, v in class_hist_raw.items()}
total_px = sum(class_hist.values())

if total_px == 0:
    raise RuntimeError("No valid pixels in sampling frame — check SB1.4-SB1.6 before proceeding.")

print("4-class pixel counts (sampling frame, snow/ice masked out, real SG boundary only):")
for cls, px in sorted(class_hist.items(), key=lambda kv: -kv[1]):
    name = BUCKET_NAMES.get(cls, f"class_{cls}")
    print(f"  {cls:>3} {name:<20} {px:>10,} px  ({px/total_px*100:5.1f}%)")


4-class pixel counts (sampling frame, snow/ice masked out, real SG boundary only):
    1 vegetation           3,526,967.403921569 px  ( 45.0%)
    2 built_up             3,127,045.835294118 px  ( 39.9%)
    4 water                825,865.415686275 px  ( 10.5%)
    3 bare                 364,979.2549019606 px  (  4.7%)


## SB1.8 — Sample allocation (vegetation oversample, floor, renormalize)

In [14]:
# --- SB1 CELL 8: Sample allocation -------------------------------------------
def allocate_sample_sizes(class_hist, total_points=TOTAL_POINTS,
                           veg_classes=OVERSAMPLE_BUCKETS,
                           oversample_factor=VEGETATION_OVERSAMPLE_FACTOR,
                           min_per_class=MIN_POINTS_PER_CLASS):
    """
    1. Weight by raw area share, boosting OVERSAMPLE_BUCKETS by oversample_factor.
    2. Renormalize weights to sum to total_points.
    3. Floor any present class at min_per_class.
    4. Rebalance +/-1 at a time (largest classes absorb the adjustment) so
       the total still equals total_points exactly.
    """
    total_px = sum(class_hist.values())
    weights = {}
    for cls, px in class_hist.items():
        share = px / total_px
        if cls in veg_classes:
            share *= oversample_factor
        weights[cls] = share

    weight_sum = sum(weights.values())
    raw_alloc = {cls: (w / weight_sum) * total_points for cls, w in weights.items()}
    floored = {cls: max(min_per_class, round(n)) for cls, n in raw_alloc.items()}

    diff = total_points - sum(floored.values())
    if diff != 0:
        order = sorted(floored, key=lambda c: floored[c], reverse=(diff < 0))
        i = 0
        while diff != 0:
            cls = order[i % len(order)]
            if diff > 0:
                floored[cls] += 1
                diff -= 1
            elif floored[cls] > 1:
                floored[cls] -= 1
                diff += 1
            i += 1
    return floored


class_alloc = allocate_sample_sizes(class_hist)

print("Sample allocation per class:")
for cls, n in sorted(class_alloc.items(), key=lambda kv: -kv[1]):
    name = BUCKET_NAMES.get(cls, f"class_{cls}")
    veg_flag = " (oversampled)" if cls in OVERSAMPLE_BUCKETS else ""
    print(f"  {cls:>3} {name:<20} {n:>4} pts{veg_flag}")
print(f"\nTotal allocated: {sum(class_alloc.values())} (target {TOTAL_POINTS})")


Sample allocation per class:
    1 vegetation            123 pts (oversampled)
    2 built_up               54 pts
    4 water                  15 pts
    3 bare                    8 pts

Total allocated: 200 (target 200)


## SB1.9 — Stratified sampling draw + shortfall check

`stratifiedSample` can silently return *fewer* points than requested for a
class if that class doesn't have enough valid pixels in-region — it doesn't
raise an error. Same "don't assume, check" discipline as Track A's zonal
join null-count check.


In [15]:
# --- SB1 CELL 9: Stratified sampling draw ------------------------------------
class_values = list(class_alloc.keys())
class_points = [class_alloc[c] for c in class_values]

samples_fc = wc_sampling_frame.stratifiedSample(
    numPoints=0,  # ignored when classPoints is set
    classBand="wc_class",
    region=sg_boundary,
    scale=TARGET_SCALE,
    classValues=class_values,
    classPoints=class_points,
    seed=RANDOM_SEED,
    geometries=True,
    dropNulls=True,
    tileScale=4,
)

# Pull once, count locally — do not re-query EE per class.
sample_records = samples_fc.getInfo()["features"]
n_drawn = len(sample_records)
print(f"Points drawn: {n_drawn} (requested {sum(class_points)})")

drawn_counts = {}
for f in sample_records:
    cls = int(f["properties"]["wc_class"])
    drawn_counts[cls] = drawn_counts.get(cls, 0) + 1

shortfalls = {}
for cls, requested in class_alloc.items():
    got = drawn_counts.get(cls, 0)
    if got < requested:
        shortfalls[cls] = (requested, got)

if shortfalls:
    print("\n⚠️  Shortfalls (requested vs actually drawn):")
    for cls, (req, got) in shortfalls.items():
        name = BUCKET_NAMES.get(cls, f"class_{cls}")
        print(f"   {name}: requested {req}, got {got}")
    print("   Likely cause: class has fewer valid pixels than requested points at this scale.")
else:
    print("\n✅ No shortfalls — every class hit its requested allocation.")


Points drawn: 200 (requested 200)

✅ No shortfalls — every class hit its requested allocation.


## SB1.10 — Verdict

In [16]:
# --- SB1 CELL 10: Verdict -------------------------------------------------------
print("\n--- SB1 Verdict ---")

sb1_checks = {
    "Sampling-frame coverage >= 90%": coverage_ok,
    "Class histogram non-empty": total_px > 0,
    f"Total points drawn == {TOTAL_POINTS}": n_drawn == TOTAL_POINTS,
    "No class shortfall > 20% below its allocation": all(
        got >= 0.8 * req for req, got in shortfalls.values()
    ) if shortfalls else True,
    "All 4 classes represented (drawn_counts > 0)": all(
        drawn_counts.get(c, 0) > 0 for c in BUCKET_NAMES if c in class_alloc
    ),
}

for check, passed in sb1_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

sb1_pass = all(sb1_checks.values())
if sb1_pass:
    print("\n✅ SB1 PASS: validation sample drawn. Proceed to build labeling table + export.")
else:
    print("\n⚠️  SB1 FAIL: fix flagged step(s) above before exporting for the labeling session.")

track_b_results["SB1_validation_sample"] = {
    "status": "PASS" if sb1_pass else "FAIL",
    "checks": sb1_checks,
    "n_points_drawn": n_drawn,
    "class_allocation": class_alloc,
    "class_shortfalls": shortfalls,
}



--- SB1 Verdict ---
  [PASS] Sampling-frame coverage >= 90%
  [PASS] Class histogram non-empty
  [PASS] Total points drawn == 200
  [PASS] No class shortfall > 20% below its allocation
  [PASS] All 4 classes represented (drawn_counts > 0)

✅ SB1 PASS: validation sample drawn. Proceed to build labeling table + export.


## SB1.11 — Build labeling table

In [17]:
# --- SB1 CELL 11: Build labeling table ----------------------------------------
import pandas as pd

rows = []
for i, f in enumerate(sample_records, start=1):
    coords = f["geometry"]["coordinates"]  # [lon, lat]
    bucket = int(f["properties"]["wc_class"])
    raw_code = int(f["properties"]["wc_raw"])
    rows.append({
        "point_id": f"P{i:04d}",
        "lon": coords[0],
        "lat": coords[1],
        "worldcover_class": bucket,
        "worldcover_class_name": BUCKET_NAMES.get(bucket, f"class_{bucket}"),
        # Audit-trail only — original fine-grained WorldCover class before
        # the 4-class collapse. Not used anywhere downstream, purely for
        # transparency if you want to check what got merged into what.
        "worldcover_class_raw": raw_code,
        "worldcover_class_raw_name": ORIGINAL_WC_NAMES.get(raw_code, f"class_{raw_code}"),
        # filled in during labeling — worldcover_class/worldcover_class_raw
        # above are reference context only, never the answer.
        "agreed_label": "",
        "confidence": "",
        "notes": "",
    })

labeling_df = pd.DataFrame(rows)
print(f"Built labeling table: {len(labeling_df)} rows")
labeling_df.head()


Built labeling table: 200 rows


,point_id,lon,lat,worldcover_class,worldcover_class_name,worldcover_class_raw,worldcover_class_raw_name,agreed_label,confidence,notes
0,P0001,103.867210,1.342852,1,vegetation,10,tree_cover,,,
1,P0002,103.822819,1.329806,1,vegetation,10,tree_cover,,,
2,P0003,103.708001,1.258203,1,vegetation,30,grassland,,,
3,P0004,103.613008,1.269821,1,vegetation,30,grassland,,,
4,P0005,103.667075,1.328279,1,vegetation,30,grassland,,,


## SB1.12 — Export (GeoJSON + CSV)

In [18]:
# --- SB1 CELL 12: Export -------------------------------------------------------
import geopandas as gpd
from shapely.geometry import Point
import os

out_dir = f"/content/drive/MyDrive/{EXPORT_FOLDER}"
os.makedirs(out_dir, exist_ok=True)

geojson_path = f"{out_dir}/{EXPORT_FILE_PREFIX}.geojson"
csv_path = f"{out_dir}/{EXPORT_FILE_PREFIX}.csv"

gdf = gpd.GeoDataFrame(
    labeling_df,
    geometry=[Point(xy) for xy in zip(labeling_df["lon"], labeling_df["lat"])],
    crs="EPSG:4326",
)
gdf.to_file(geojson_path, driver="GeoJSON")
labeling_df.to_csv(csv_path, index=False)

print(f"Wrote {len(gdf)} points to:")
print(f"  {geojson_path}")
print(f"  {csv_path}")
print("\nClass distribution in final sample:")
print(labeling_df["worldcover_class_name"].value_counts())


Wrote 200 points to:
  /content/drive/MyDrive/urban_heat_sg/validation_sample_200.geojson
  /content/drive/MyDrive/urban_heat_sg/validation_sample_200.csv

Class distribution in final sample:
worldcover_class_name
vegetation    123
built_up       54
water          15
bare            8
Name: count, dtype: int64


## SB1.13 (optional) — Reload check from mounted Drive

In [19]:
# --- SB1 CELL 13 (optional): Reload check --------------------------------------
check_df = pd.read_csv(csv_path)
print(f"Reloaded {len(check_df)} rows from {csv_path}")
assert len(check_df) == TOTAL_POINTS, "Row count mismatch on reload — investigate before labeling."
check_df.head()


Reloaded 200 rows from /content/drive/MyDrive/urban_heat_sg/validation_sample_200.csv


,point_id,lon,lat,worldcover_class,worldcover_class_name,worldcover_class_raw,worldcover_class_raw_name,agreed_label,confidence,notes
0,P0001,103.867210,1.342852,1,vegetation,10,tree_cover,NaN,NaN,NaN
1,P0002,103.822819,1.329806,1,vegetation,10,tree_cover,NaN,NaN,NaN
2,P0003,103.708001,1.258203,1,vegetation,30,grassland,NaN,NaN,NaN
3,P0004,103.613008,1.269821,1,vegetation,30,grassland,NaN,NaN,NaN
4,P0005,103.667075,1.328279,1,vegetation,30,grassland,NaN,NaN,NaN
